<a href="https://colab.research.google.com/github/ADCO02/trabajoAprendizajeProfundo/blob/main/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Trabajo Aprendizaje Profundo

Ruben García y Adrian de Celis

## Objetivo del trabajo

El objetivo de este trabajo es estudiar cómo pueden aplicarse las técnicas de aprendizaje profundo a la generación de voz, centrándonos en los sistemas modernos de text-to-speech y, en particular, en los modelos capaces de realizar síntesis multilingüe y clonación de voz. Para ello se utilizará la biblioteca Coqui TTS como marco práctico y se analizará el modelo XTTS como caso de estudio representativo.

A lo largo del trabajo se explicará, primero, el fundamento teórico de la generación de voz mediante redes neuronales profundas, describiendo las principales etapas que intervienen en un sistema TTS moderno. Después, se estudiará cómo estas ideas se materializan en una librería real de uso práctico, mostrando qué tipos de modelos incluye Coqui TTS y qué capacidades ofrece para generar voz natural a partir de texto.

Además de la parte teórica, el trabajo incorporará una implementación en Python orientada a la inferencia con modelos preentrenados, así como una serie de experimentos para analizar la calidad de la voz generada, la capacidad de clonación de voz a partir de un audio de referencia y el comportamiento del sistema en distintos idiomas. Finalmente, se discutirán las limitaciones técnicas, los posibles sesgos y las implicaciones éticas derivadas del uso de este tipo de tecnologías.

En conjunto, el propósito del trabajo no es solo mostrar que un modelo puede convertir texto en audio, sino entender qué componentes hacen posible esa generación, qué grado de control ofrecen los modelos actuales sobre la identidad y el estilo de la voz y qué retos técnicos y sociales plantea esta tecnología.

## TTS y voice generation

El término TTS (Text-to-Speech) hace referencia a los sistemas capaces de transformar texto escrito en una señal de voz sintética. El objetivo de estos sistemas es producir un audio inteligible y natural, respetando tanto el contenido lingüístico del texto como propiedades prosódicas como el ritmo, la entonación, las pausas o el énfasis.

Tradicionalmente, la síntesis de voz se abordaba mediante sistemas basados en reglas o mediante síntesis concatenativa, en los que se unían fragmentos de voz previamente grabados. Sin embargo, estos enfoques tenían limitaciones importantes en naturalidad, flexibilidad y capacidad de generalización. La aparición del aprendizaje profundo supuso un cambio significativo, ya que permitió entrenar modelos capaces de aprender directamente la relación entre texto y habla a partir de grandes cantidades de datos.

Dentro de este contexto, el concepto de voice generation puede entenderse como un término más amplio que engloba distintas tareas relacionadas con la generación artificial de voz. Entre ellas se encuentran:

* la síntesis de voz propiamente dicha, donde el sistema genera audio a partir de texto;
* la clonación de voz, donde además se intenta que la voz generada conserve las características de un hablante concreto a partir de una muestra de referencia;
* y la conversión de voz, donde una grabación de un hablante se transforma para que suene como la de otro.

Por tanto, mientras que TTS describe de forma específica la tarea de convertir texto en habla, voice generation abarca un conjunto más amplio de técnicas destinadas a crear, modificar o controlar la voz sintética. En los modelos más recientes, estas tareas ya no están completamente separadas, sino que se combinan en sistemas que permiten controlar simultáneamente el contenido del mensaje, el idioma y la identidad vocal del hablante.

En este trabajo se pondrá especial atención en los modelos que no solo generan voz a partir de texto, sino que además permiten condicionar la salida con una voz de referencia, lo que abre la puerta a aplicaciones como la personalización de asistentes virtuales, la localización de contenidos en múltiples idiomas o la accesibilidad para usuarios con necesidades específicas.

## Arquitectura TTS

Aunque los sistemas de síntesis de voz han evolucionado mucho en los últimos años, la mayoría de arquitecturas modernas comparten una estructura general compuesta por varias etapas. De forma simplificada, un sistema TTS recibe una secuencia de texto como entrada y produce una señal de audio como salida, pero entre ambos extremos suelen existir representaciones intermedias que facilitan el aprendizaje del proceso.

La primera etapa consiste en el procesamiento del texto. Aquí el texto original se normaliza y transforma en una representación adecuada para el modelo. Esta fase puede incluir tareas como expandir abreviaturas, convertir números a palabras, gestionar signos de puntuación o transformar caracteres en unidades fonéticas o subpalabras. El objetivo es ofrecer al sistema una representación lingüística que capture con claridad qué debe pronunciarse.

A continuación, muchos sistemas generan una representación acústica intermedia, normalmente un mel-spectrograma. Esta representación resume cómo evoluciona la energía de la señal en distintas bandas de frecuencia a lo largo del tiempo y resulta más sencilla de modelar que la forma de onda cruda. En otras palabras, el modelo aprende primero a “imaginar” la estructura acústica de la frase antes de convertirla en sonido real.

La siguiente etapa es el vocoder, que se encarga de transformar esa representación acústica en una señal de audio final. El vocoder tiene un papel crucial en la calidad perceptiva de la voz generada, ya que de él dependen aspectos como la naturalidad, la nitidez y la ausencia de artefactos. En los sistemas modernos, tanto el predictor acústico como el vocoder suelen estar implementados mediante redes neuronales profundas especializadas.

En modelos más avanzados, especialmente los de tipo multi-speaker o multilingual, se añaden también mecanismos de condicionamiento. Esto significa que, además del texto, el modelo recibe información adicional, como por ejemplo:

* la identidad del hablante,
* el idioma en que debe sintetizar,
* el estilo de pronunciación,
* o un embedding extraído de un audio de referencia.

Este condicionamiento permite controlar características de la voz generada sin modificar el contenido textual. Así, dos entradas con el mismo texto pueden dar lugar a audios muy distintos si cambia el hablante o el idioma especificado.

Desde el punto de vista conceptual, un sistema TTS moderno puede resumirse así:

texto → representación lingüística → representación acústica → vocoder → audio

Cuando además se quiere clonar la voz de una persona concreta, se incorpora una señal adicional:

texto + información del hablante → síntesis condicionada → audio con identidad vocal específica

Esta arquitectura modular ayuda a entender por qué la generación de voz es un problema complejo: no basta con pronunciar correctamente las palabras, sino que también hay que modelar el tiempo, la prosodia, el timbre y, en muchos casos, la identidad del hablante. Precisamente, uno de los principales avances del deep learning en este campo ha sido lograr que todos estos factores se aprendan de forma conjunta y con una calidad cada vez más cercana a la voz humana.

## Coqui TTS

Coqui TTS es una biblioteca de código abierto orientada a la síntesis de voz mediante aprendizaje profundo. Su principal interés en el contexto de este trabajo es que proporciona una base unificada para utilizar distintos modelos de generación de voz, realizar inferencia con modelos preentrenados y, en entornos más avanzados, entrenar o ajustar modelos propios.

Una de las fortalezas de Coqui TTS es que no se limita a ofrecer un único sistema cerrado, sino que reúne varias piezas del ecosistema de síntesis de voz. Dentro de la biblioteca pueden encontrarse modelos de predicción acústica, vocoders neuronales, mecanismos de codificación del hablante y utilidades relacionadas con el análisis y procesamiento de conjuntos de datos. Esto la convierte en una herramienta útil tanto para fines docentes como para experimentación aplicada.

Desde un punto de vista práctico, Coqui TTS facilita trabajar con distintas configuraciones de generación de voz, por ejemplo:

* modelos de un solo hablante;
* modelos multihablante;
* modelos multilingües;
* sistemas de clonación de voz;
* y tareas próximas, como la conversión de voz.

Este carácter modular hace que la librería sea especialmente interesante para explicar la generación de voz en un entorno académico. En lugar de tratar la síntesis de voz como una “caja negra”, Coqui TTS permite mostrar que detrás del audio final existen distintos componentes especializados y diferentes enfoques de modelado.

Otra ventaja importante es su orientación a la reproducibilidad y a la facilidad de uso. La librería permite cargar modelos preentrenados desde Python con pocas líneas de código, generar audios a partir de texto y experimentar con parámetros como el idioma o la voz de referencia. Esto reduce la complejidad de la implementación y permite centrar el trabajo en el análisis conceptual y experimental, sin necesidad de entrenar un modelo desde cero.

Para este trabajo, Coqui TTS se utilizará como entorno principal de experimentación porque permite ilustrar de forma clara la evolución de la síntesis de voz moderna: desde modelos TTS convencionales hasta sistemas más avanzados capaces de sintetizar en varios idiomas y aproximar el timbre de un hablante concreto a partir de un ejemplo de voz.

## XTTS y generación multilingüe

Dentro del ecosistema de Coqui TTS, uno de los modelos más interesantes para estudiar la generación de voz es XTTS, ya que combina varias de las capacidades más relevantes de los sistemas actuales: síntesis de voz a partir de texto, funcionamiento multilingüe y clonación de voz condicionada por una muestra de referencia.

La idea central de este tipo de modelos es que la voz generada no depende únicamente del texto de entrada. Además del contenido lingüístico, el sistema puede recibir información sobre qué idioma debe utilizar y cómo debe sonar la voz resultante. Para ello, se emplea un audio de referencia del hablante objetivo, a partir del cual el modelo extrae información relacionada con el timbre, el estilo y otras características vocales. De este modo, el sistema puede generar una frase nueva, incluso en otro idioma, manteniendo de forma aproximada la identidad vocal del hablante de referencia.

Esta capacidad representa un avance significativo respecto a sistemas TTS más clásicos. En un modelo convencional, la voz suele estar fijada por el propio entrenamiento y el usuario únicamente controla el texto. En cambio, en un sistema como XTTS el proceso de síntesis está condicionado por varios factores a la vez:

* el contenido textual,
* el idioma de salida,
* y una representación del hablante obtenida a partir de un ejemplo de audio.

Gracias a este diseño, XTTS resulta especialmente adecuado para estudiar dos problemas de gran interés actual. El primero es la síntesis multilingüe, donde un mismo sistema puede generar voz en distintos idiomas sin necesidad de entrenar un modelo independiente para cada uno. El segundo es la clonación de voz de pocos ejemplos, donde se intenta reproducir la identidad vocal de una persona a partir de una muestra relativamente breve.

Desde el punto de vista académico, XTTS permite ilustrar muy bien cómo el aprendizaje profundo ha ampliado el alcance del TTS. Ya no se trata solo de producir voz inteligible, sino de dotar al modelo de mecanismos de control sobre atributos que antes eran difíciles de modificar, como la identidad del hablante o la transferencia entre idiomas. Esto convierte a XTTS en un caso de estudio especialmente representativo de la generación de voz moderna.

Sin embargo, estas capacidades también introducen nuevos retos. La calidad de la clonación puede depender de la limpieza del audio de referencia, de la duración de la muestra o de la similitud entre idiomas. Además, aunque el modelo pueda producir resultados muy convincentes, esto no implica una reproducción perfecta de la voz original. En la práctica, la salida suele ser una aproximación que conserva algunos rasgos característicos del hablante, pero que puede mostrar variaciones en prosodia, pronunciación o estabilidad.

En este trabajo, XTTS se utilizará como ejemplo principal para analizar cómo un sistema de deep learning puede generar voz natural, multilingüe y condicionada por una identidad vocal, permitiendo estudiar tanto sus posibilidades técnicas como sus limitaciones reales.

## Implementación

Versión python en colab: 2025.07

In [2]:
# 1. Limpiamos las versiones incompatibles que trae Colab por defecto
!pip uninstall -y torch torchaudio torchvision
!pip uninstall -y numpy TTS transformers

# 2. Instalamos el ecosistema perfecto y estable para XTTS
!pip install -q torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1
!pip install -q numpy==1.26.4 transformers==4.45.2
!pip install -q TTS==0.22.0

print("¡Instalación limpia y completada!")

Found existing installation: torch 2.6.0+cu124
Uninstalling torch-2.6.0+cu124:
  Successfully uninstalled torch-2.6.0+cu124
Found existing installation: torchaudio 2.6.0+cu124
Uninstalling torchaudio-2.6.0+cu124:
  Successfully uninstalled torchaudio-2.6.0+cu124
Found existing installation: torchvision 0.21.0+cu124
Uninstalling torchvision-0.21.0+cu124:
  Successfully uninstalled torchvision-0.21.0+cu124
Found existing installation: numpy 2.0.2
Uninstalling numpy-2.0.2:
  Successfully uninstalled numpy-2.0.2
Found existing installation: transformers 4.53.2
Uninstalling transformers-4.53.2:
  Successfully uninstalled transformers-4.53.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 906.5/906.5 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 131.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 102.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Reiniciar sesión

In [1]:
import os
from pathlib import Path

from IPython.display import Audio, display
from TTS.api import TTS

In [2]:
BASE_DIR = Path("trabajo_tts")
REF_DIR = BASE_DIR / "referencias"
OUT_DIR = BASE_DIR / "salidas"

REF_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Carpeta de referencias:", REF_DIR.resolve())
print("Carpeta de salidas:", OUT_DIR.resolve())

Carpeta de referencias: /content/trabajo_tts/referencias
Carpeta de salidas: /content/trabajo_tts/salidas


In [3]:
import torch
print(torch.__version__)

2.5.1+cu124


In [4]:
model_name = "tts_models/multilingual/multi-dataset/xtts_v2"
tts = TTS(model_name)
print("Modelo cargado:", model_name)

 > You must confirm the following:
 | > "I have purchased a commercial license from Coqui: licensing@coqui.ai"
 | > "Otherwise, I agree to the terms of the non-commercial CPML: https://coqui.ai/cpml" - [y/n]
 | | > y
 > Downloading model to /root/.local/share/tts/tts_models--multilingual--multi-dataset--xtts_v2


100%|█████████▉| 1.86G/1.87G [00:22<00:00, 93.6MiB/s]
100%|██████████| 1.87G/1.87G [00:22<00:00, 82.0MiB/s]
4.37kiB [00:00, 15.7kiB/s]

361kiB [00:00, 1.19MiB/s]
100%|██████████| 32.0/32.0 [00:00<00:00, 108iB/s]


 > Model's license - CPML
 > Check https://coqui.ai/cpml.txt for more info.
 > Using model: xtts


/usr/local/lib/python3.11/dist-packages/TTS/tts/layers/xtts/xtts_manager.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.speakers = torch.load(speaker_file_path)
/u

Modelo cargado: tts_models/multilingual/multi-dataset/xtts_v2


In [5]:
import os
from pathlib import Path
import urllib.request

REF_DIR = Path("trabajo_tts/referencias")
REF_DIR.mkdir(parents=True, exist_ok=True)

urls = {
    "voz_referencia_1.wav": "https://huggingface.co/voices/VCTK_British_English_Females/resolve/main/samples/VCTK_p225.wav",
    "voz_referencia_2.wav": "https://huggingface.co/voices/VCTK_British_English_Females/resolve/main/samples/VCTK_p228.wav",
    "voz_referencia_3.wav": "https://huggingface.co/voices/VCTK_British_English_Females/resolve/main/samples/VCTK_p229.wav",
    "voz_referencia_4.wav": "https://huggingface.co/voices/VCTK_British_English_Females/resolve/main/samples/VCTK_p230.wav",
    "voz_referencia_5.wav": "https://huggingface.co/voices/VCTK_British_English_Females/resolve/main/samples/VCTK_p231.wav",
}

for nombre, url in urls.items():
    destino = REF_DIR / nombre
    urllib.request.urlretrieve(url, destino)
    print("Descargado:", destino)

print("\nArchivos disponibles:")
for f in sorted(REF_DIR.iterdir()):
    print("-", f.name)

Descargado: trabajo_tts/referencias/voz_referencia_1.wav
Descargado: trabajo_tts/referencias/voz_referencia_2.wav
Descargado: trabajo_tts/referencias/voz_referencia_3.wav
Descargado: trabajo_tts/referencias/voz_referencia_4.wav
Descargado: trabajo_tts/referencias/voz_referencia_5.wav

Archivos disponibles:
- voz_referencia_1.wav
- voz_referencia_2.wav
- voz_referencia_3.wav
- voz_referencia_4.wav
- voz_referencia_5.wav


In [6]:
speaker_wav = REF_DIR / "voz_referencia_1.wav"

print("Ruta esperada del archivo de referencia:")
print(speaker_wav.resolve())

Ruta esperada del archivo de referencia:
/content/trabajo_tts/referencias/voz_referencia_1.wav


In [7]:
texto_es = "Hola, esta es una prueba de generación de voz con aprendizaje profundo."
salida_es = OUT_DIR / "ejemplo_es.wav"

tts.tts_to_file(
    text=texto_es,
    file_path=str(salida_es),
    speaker_wav=str(speaker_wav),
    language="es"
)

print("Archivo generado en:", salida_es)
display(Audio(str(salida_es)))

 > Text splitted to sentences.
['Hola, esta es una prueba de generación de voz con aprendizaje profundo.']


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


 > Processing time: 41.40209364891052
 > Real-time factor: 7.586392808124559
Archivo generado en: trabajo_tts/salidas/ejemplo_es.wav


In [8]:
texto_en = "Hello, this is a multilingual voice generation test using a cloned reference voice."
salida_en = OUT_DIR / "ejemplo_en.wav"

tts.tts_to_file(
    text=texto_en,
    file_path=str(salida_en),
    speaker_wav=str(speaker_wav),
    language="en"
)

print("Archivo generado en:", salida_en)
display(Audio(str(salida_en)))

 > Text splitted to sentences.
['Hello, this is a multilingual voice generation test using a cloned reference voice.']
 > Processing time: 55.30535006523132
 > Real-time factor: 7.991788356783781
Archivo generado en: trabajo_tts/salidas/ejemplo_en.wav


## Experimentos

In [9]:
def generar_audio(texto, idioma, nombre_salida, speaker_path):
    ruta_salida = OUT_DIR / nombre_salida
    tts.tts_to_file(
        text=texto,
        file_path=str(ruta_salida),
        speaker_wav=str(speaker_path),
        language=idioma
    )
    return ruta_salida

In [10]:
# Mismo hablante, distintos idiomas

textos = {
    "es": "Este experimento evalúa la capacidad del modelo para mantener la identidad vocal en distintos idiomas.",
    "en": "This experiment evaluates whether the model preserves speaker identity across different languages.",
    "fr": "Cette expérience évalue si le modèle conserve l'identité vocale dans plusieurs langues."
}

rutas = {}
for idioma, texto in textos.items():
    nombre = f"exp1_{idioma}.wav"
    rutas[idioma] = generar_audio(texto, idioma, nombre, speaker_wav)

for idioma, ruta in rutas.items():
    print(f"{idioma}: {ruta}")
    display(Audio(str(ruta)))

 > Text splitted to sentences.
['Este experimento evalúa la capacidad del modelo para mantener la identidad vocal en distintos idiomas.']
 > Processing time: 64.48029732704163
 > Real-time factor: 8.350898388669227
 > Text splitted to sentences.
['This experiment evaluates whether the model preserves speaker identity across different languages.']
 > Processing time: 51.392674684524536
 > Real-time factor: 8.211655628940333
 > Text splitted to sentences.
["Cette expérience évalue si le modèle conserve l'identité vocale dans plusieurs langues."]
 > Processing time: 38.278595209121704
 > Real-time factor: 8.059998322776295
es: trabajo_tts/salidas/exp1_es.wav


en: trabajo_tts/salidas/exp1_en.wav


fr: trabajo_tts/salidas/exp1_fr.wav


In [11]:
# Texto corto vs largo

texto_corto = "Hoy hace buen día."
texto_largo = (
    "Hoy hace buen día y vamos a aprovechar esta prueba para observar cómo cambia la entonación "
    "del modelo cuando la secuencia de entrada es más larga y contiene varias pausas naturales, "
    "comas y una estructura sintáctica algo más compleja."
)

ruta_corta = generar_audio(texto_corto, "es", "exp2_corto.wav", speaker_wav)
ruta_larga = generar_audio(texto_largo, "es", "exp2_largo.wav", speaker_wav)

print("Texto corto")
display(Audio(str(ruta_corta)))

print("Texto largo")
display(Audio(str(ruta_larga)))

 > Text splitted to sentences.
['Hoy hace buen día.']
 > Processing time: 13.726295471191406
 > Real-time factor: 8.039333168820933
 > Text splitted to sentences.
['Hoy hace buen día y vamos a aprovechar esta prueba para observar cómo cambia la entonación del modelo cuando la secuencia de entrada es más larga y contiene varias pausas naturales, comas y una estructura sintáctica algo más compleja.']
 > Processing time: 119.206063747406
 > Real-time factor: 7.971316250273857
Texto corto


Texto largo


In [12]:
# Nombres propios y pronunciación

texto_nombres = (
    "María visitó OpenAI en San Francisco y después presentó su proyecto en la Universidad de Zaragoza."
)

ruta_nombres = generar_audio(texto_nombres, "es", "exp3_nombres.wav", speaker_wav)

print("Prueba de pronunciación de nombres propios:")
display(Audio(str(ruta_nombres)))

 > Text splitted to sentences.
['María visitó OpenAI en San Francisco y después presentó su proyecto en la Universidad de Zaragoza.']
 > Processing time: 49.09251666069031
 > Real-time factor: 7.659416339070964
Prueba de pronunciación de nombres propios:


In [13]:
# Distintas voces de referencias

speaker_wav_1 = REF_DIR / "voz_referencia_1.wav"
speaker_wav_2 = REF_DIR / "voz_referencia_2.wav"

texto_comp = "Esta frase se utilizará para comparar dos voces de referencia diferentes con el mismo texto."

ruta_ref1 = generar_audio(texto_comp, "es", "exp4_ref1.wav", speaker_wav_1)

print("Resultado con la primera referencia:")
display(Audio(str(ruta_ref1)))

if speaker_wav_2.exists():
    ruta_ref2 = generar_audio(texto_comp, "es", "exp4_ref2.wav", speaker_wav_2)
    print("Resultado con la segunda referencia:")
    display(Audio(str(ruta_ref2)))
else:
    print("No se encontró 'voz_referencia_2.wav'. Sube un segundo audio si quieres ejecutar esta comparación.")

 > Text splitted to sentences.
['Esta frase se utilizará para comparar dos voces de referencia diferentes con el mismo texto.']
 > Processing time: 54.44405746459961
 > Real-time factor: 7.933881431045927
Resultado con la primera referencia:


 > Text splitted to sentences.
['Esta frase se utilizará para comparar dos voces de referencia diferentes con el mismo texto.']
 > Processing time: 58.41773343086243
 > Real-time factor: 7.9231314717456245
Resultado con la segunda referencia:


## Conclusiones

A lo largo de este trabajo se ha podido comprobar que las técnicas actuales de aprendizaje profundo aplicadas a la síntesis de voz permiten obtener resultados muy avanzados y naturales, especialmente cuando se emplean modelos modernos como XTTS v2 dentro del ecosistema de Coqui TTS. La experiencia práctica ha servido para entender no solo la estructura general de un sistema TTS, sino también sus posibilidades reales en tareas de generación multilingüe y clonación de voz.

Los experimentos realizados muestran que el modelo es capaz de mantener de forma razonable la identidad vocal al generar audio en distintos idiomas a partir de una misma voz de referencia. Esto confirma una de las principales fortalezas de XTTS: su capacidad multilingüe, que permite reutilizar un mismo perfil de hablante en contextos lingüísticos diferentes sin necesidad de entrenar un modelo específico para cada idioma.

También se ha observado que la longitud y complejidad del texto de entrada influyen en el resultado final. En frases cortas, la generación suele ser más directa y estable, mientras que en secuencias más largas aparecen más matices de entonación y pausas, lo que permite apreciar mejor el potencial expresivo del modelo, pero también evidencia que la síntesis depende en gran medida de cómo esté estructurado el texto.

Otro aspecto relevante ha sido la prueba con nombres propios y términos específicos, que pone de manifiesto que, aunque la pronunciación generada es en general convincente, este tipo de elementos sigue siendo un reto importante en los sistemas TTS. La correcta pronunciación de nombres, marcas o lugares depende del idioma, del contexto y del propio entrenamiento del modelo, por lo que sigue siendo un punto donde pueden aparecer limitaciones.

Por último, la comparación entre distintas voces de referencia demuestra que el sistema puede adaptar la salida a características vocales diferentes, lo que refuerza su utilidad en aplicaciones de personalización de voz. Sin embargo, la calidad final sigue dependiendo de factores como la claridad del audio de referencia, su duración y las condiciones de grabación.

En conjunto, puede concluirse que los sistemas actuales de generación de voz basados en aprendizaje profundo ofrecen un nivel de calidad muy alto y abren la puerta a numerosas aplicaciones en accesibilidad, asistentes virtuales, doblaje, educación o creación de contenidos. Aun así, persisten desafíos relacionados con la pronunciación precisa, la expresividad y el control fino sobre la voz generada. Como trabajo futuro, sería interesante realizar una evaluación más sistemática con métricas objetivas y pruebas perceptuales con usuarios, para comparar de forma más rigurosa la calidad y naturalidad de las voces sintetizadas.

## Hugginface

Enlace al espacio:
[https://huggingface.co/spaces/adcelis/AP-xtts-demo](https://huggingface.co/spaces/adcelis/AP-xtts-demo)